# Dataset25: Detailed LightGBM Model Analysis

This notebook provides a detailed evaluation of LTE and 5G NR signal-quality classification using the final feature-engineered dataset, Dataset25.

Dataset25 contains 4,985 labelled-union grid cells and 35 predictors covering environmental, socioeconomic, cellular-infrastructure, interaction, remote-sensing, and geographic information.

LTE and 5G NR are evaluated separately using LightGBM and stratified five-fold cross-validation. Out-of-fold predictions are used to examine overall performance, class-specific performance, confusion matrices, and feature importance.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Modelling and Evaluation)
- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Table 4.5 (Overall and Poor-Class Performance of LightGBM)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
from pathlib import Path

REPOSITORY_ROOT = Path(
    "/content/drive/MyDrive/england-lte-5g-signal-prediction"
)
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from config import FIGURES_DIR, PROCESSED_DATA_DIR


## 1. Environment and Evaluation Metrics

The required data-processing, modelling, and visualisation libraries are imported.

Performance is evaluated using accuracy, balanced accuracy, Macro-F1, Matthews Correlation Coefficient, and class-specific precision, recall, and F1-score.

A fixed random state and five cross-validation folds are used for reproducibility.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Modelling and Evaluation)
- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Appendix C, Table C.1 (Reproducibility Summary)

In [ ]:
# ============================================================
# Cell 1. Imports
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from lightgbm import LGBMClassifier

RANDOM_STATE = 42
N_SPLITS = 5

## 2. Load and Prepare Dataset25

Dataset25 is loaded and validated for row count, unique grid identifiers, duplicate records, and LTE and 5G NR label availability.

LTE and 5G NR modelling subsets are prepared separately. Target-derived RSRP statistics, Ofcom measurement counts, grid identifiers, and geometry are excluded from the predictor matrix to prevent target leakage.

The preparation procedure also verifies that all 35 predictors are numeric and contain no missing or infinite values.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Data Preparation)
- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Appendix C, Table C.2 (Definitions of the 35 Predictors)

In [ ]:
# ============================================================
# Cell 2. Load and Validate Dataset25
# ============================================================

path = str(PROCESSED_DATA_DIR / "dataset25_final.csv")

df_original = pd.read_csv(
    path,
    low_memory=False
)

print("Dataset25 shape:", df_original.shape)
print("Unique grid IDs:", df_original["grid_id"].nunique())
print("Duplicate grid IDs:", df_original["grid_id"].duplicated().sum())

print("\nLTE labelled grids:")
print(df_original["lte_signal_class"].notna().sum())

print("\n5G NR labelled grids:")
print(df_original["nr_signal_class"].notna().sum())

In [ ]:
# ============================================================
# Cell 3. Prepare Dataset25
# ============================================================

def prepare_dataset25(target_col):

    valid_classes = [
        "Excellent",
        "Good",
        "Poor"
    ]

    # Sort to ensure identical and reproducible fold allocation
    df = (
        df_original
        .sort_values("grid_id")
        .reset_index(drop=True)
        .copy()
    )

    # --------------------------------------------------------
    # 1. Retain valid labelled grids
    # --------------------------------------------------------

    df[target_col] = (
        df[target_col]
        .astype("string")
        .str.strip()
    )

    df = (
        df[
            df[target_col].isin(valid_classes)
        ]
        .copy()
        .reset_index(drop=True)
    )

    detected_classes = set(
        df[target_col].dropna().unique()
    )

    if detected_classes != set(valid_classes):
        raise ValueError(
            f"Unexpected target classes: {detected_classes}"
        )

    # --------------------------------------------------------
    # 2. Remove identifiers and target-derived columns
    # --------------------------------------------------------

    drop_cols = [
        "geometry",
        "grid_id",

        "lte_signal_class",
        "nr_signal_class",

        "lte_min_rsrp",
        "lte_mean_rsrp",
        "lte_median_rsrp",
        "lte_point_count",

        "nr_min_rsrp",
        "nr_mean_rsrp",
        "nr_median_rsrp",
        "nr_point_count"
    ]

    feature_cols = [
        column
        for column in df.columns
        if column not in drop_cols
    ]

    non_numeric_cols = [
        column
        for column in feature_cols
        if not pd.api.types.is_numeric_dtype(df[column])
    ]

    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric predictors found: {non_numeric_cols}"
        )

    X = df[feature_cols].copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    if X.isna().any().any():
        missing_cols = X.columns[
            X.isna().any()
        ].tolist()

        raise ValueError(
            f"Missing predictor values found: {missing_cols}"
        )

    if X.shape[1] != 35:
        raise ValueError(
            f"Expected 35 predictors, but found {X.shape[1]}."
        )

    # --------------------------------------------------------
    # 3. Encode target
    # --------------------------------------------------------

    y = df[target_col].copy()

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    print("=" * 70)
    print("Target:", target_col)
    print("=" * 70)
    print("Rows:", len(df))
    print("Features:", X.shape[1])

    print("\nClass distribution:")
    print(
        y.value_counts()
        .reindex(valid_classes)
    )

    print("\nLabel encoding:")
    print(
        dict(
            zip(
                label_encoder.classes_,
                label_encoder.transform(
                    label_encoder.classes_
                )
            )
        )
    )

    return (
        X,
        y_encoded,
        label_encoder
    )

## 3. LightGBM and Out-of-Fold Evaluation

The LightGBM classifier uses the same fixed configuration employed in the progressive feature-set comparison.

Stratified five-fold cross-validation generates one out-of-fold prediction for every labelled grid. Overall and fold-specific metrics are calculated, and gain-based feature importance is collected from each fitted model.

Mean feature importance across the five folds is used to reduce dependence on a single fitted model.

### ※ Related dissertation sections

- Section 3.1 (Research Approach—Modelling and Evaluation)
- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Appendix C, Table C.1 (Reproducibility Summary)

In [ ]:
# ============================================================
# Cell 4. Baseline LightGBM Definition
# Same configuration as Dataset19–25 comparison
# ============================================================

def create_lgbm():

    return LGBMClassifier(
        n_estimators=100,
        max_depth=10,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        verbose=-1,
        n_jobs=-1
    )

In [ ]:
# ============================================================
# Cell 5. Define 5-Fold OOF Evaluation Function
# ============================================================

def run_oof_evaluation(
    X,
    y,
    le,
    network_name
):

    cv = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    # Use -1 so unassigned predictions can be detected
    oof_pred = np.full(
        len(y),
        -1,
        dtype=int
    )

    fold_results = []
    fold_importances = []

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        print(
            f"{network_name} | Fold {fold}/{N_SPLITS}"
        )

        X_train = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()

        y_train = y[train_idx]
        y_val = y[val_idx]

        # Train baseline LightGBM
        model = create_lgbm()

        model.fit(
            X_train,
            y_train
        )

        # Generate validation predictions
        y_pred = model.predict(
            X_val
        ).astype(int)

        oof_pred[val_idx] = y_pred

        # Store fold-level metrics
        fold_results.append({
            "Fold": fold,
            "Accuracy": accuracy_score(
                y_val,
                y_pred
            ),
            "Balanced Accuracy": balanced_accuracy_score(
                y_val,
                y_pred
            ),
            "Macro-F1": f1_score(
                y_val,
                y_pred,
                average="macro",
                zero_division=0
            ),
            "MCC": matthews_corrcoef(
                y_val,
                y_pred
            )
        })

        # Gain-based feature importance
        fold_importances.append(
            model.booster_.feature_importance(
                importance_type="gain"
            )
        )

    # Ensure every row received one OOF prediction
    if (oof_pred == -1).any():
        raise ValueError(
            "Some observations did not receive an OOF prediction."
        )

    fold_results_df = pd.DataFrame(
        fold_results
    )

    # Pooled OOF metrics
    overall_results = {
        "Network": network_name,
        "Accuracy": accuracy_score(
            y,
            oof_pred
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y,
            oof_pred
        ),
        "Macro-F1": f1_score(
            y,
            oof_pred,
            average="macro",
            zero_division=0
        ),
        "MCC": matthews_corrcoef(
            y,
            oof_pred
        ),
        "CV Macro-F1 Mean": (
            fold_results_df["Macro-F1"].mean()
        ),
        "CV Macro-F1 SD": (
            fold_results_df["Macro-F1"].std(ddof=0)
        )
    }

    # Average gain importance across folds
    mean_importance = np.mean(
        fold_importances,
        axis=0
    )

    feature_importance_df = pd.DataFrame({
        "Feature": X.columns,
        "Importance": mean_importance
    })

    # Convert raw gain to percentage contribution
    total_importance = (
        feature_importance_df["Importance"].sum()
    )

    if total_importance > 0:
        feature_importance_df["Importance (%)"] = (
            feature_importance_df["Importance"]
            / total_importance
            * 100
        )

    feature_importance_df = (
        feature_importance_df
        .sort_values(
            "Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return (
        overall_results,
        fold_results_df,
        oof_pred,
        feature_importance_df
    )

## 4. LTE and 5G NR Model Evaluation

The evaluation procedure is run separately for LTE and 5G NR.

For each network, the notebook reports fold-level results, pooled out-of-fold performance, mean cross-validation Macro-F1, and variation across the five folds.

### ※ Related dissertation sections

- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Table 4.5 (Overall and Poor-Class Performance of LightGBM)

In [ ]:
# ============================================================
# Cell 6. LTE 5-Fold OOF Analysis
# ============================================================

X_lte, y_lte, le_lte = prepare_dataset25(
    "lte_signal_class"
)

(
    lte_overall,
    lte_folds,
    lte_oof_pred,
    lte_importance
) = run_oof_evaluation(
    X=X_lte,
    y=y_lte,
    le=le_lte,
    network_name="LTE"
)

print("\nLTE Fold Results")
display(
    lte_folds.round(4)
)

print("\nLTE Overall OOF Results")
display(
    pd.DataFrame(
        [lte_overall]
    ).round(4)
)

In [ ]:
# ============================================================
# Cell 7. 5G NR 5-Fold OOF Analysis
# ============================================================

X_nr, y_nr, le_nr = prepare_dataset25(
    "nr_signal_class"
)

(
    nr_overall,
    nr_folds,
    nr_oof_pred,
    nr_importance
) = run_oof_evaluation(
    X=X_nr,
    y=y_nr,
    le=le_nr,
    network_name="5G NR"
)

print("\n5G NR Fold Results")
display(
    nr_folds.round(4)
)

print("\n5G NR Overall OOF Results")
display(
    pd.DataFrame(
        [nr_overall]
    ).round(4)
)

## 5. Overall and Class-Specific Performance

The LTE and 5G NR results are combined into an overall performance summary.

Classification reports provide precision, recall, and F1-score for the Excellent, Good, and Poor classes. Poor-class metrics are also extracted separately because weak-signal identification is a central focus of the study.

### ※ Related dissertation sections

- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Table 4.5 (Overall and Poor-Class Performance of LightGBM)
- Section 5.1 (Overall Classification Performance)
- Section 5.2 (Class Imbalance and Poor-Signal Identification)

In [ ]:
# ============================================================
# Cell 8. Dataset25 LightGBM OOF Performance Summary
# ============================================================

performance_summary = pd.DataFrame([
    lte_overall,
    nr_overall
])

summary_cols = [
    "Network",
    "Accuracy",
    "Balanced Accuracy",
    "Macro-F1",
    "MCC",
    "CV Macro-F1 Mean",
    "CV Macro-F1 SD"
]

performance_summary = performance_summary[
    [
        col for col in summary_cols
        if col in performance_summary.columns
    ]
]

print("Dataset25 LightGBM OOF Performance Summary")

display(
    performance_summary.round(4)
)

In [ ]:
# ============================================================
# Cell 9. OOF Classification Reports
# ============================================================

def display_classification_report(
    y_true,
    y_pred,
    label_encoder,
    network_name
):
    class_order = [
        "Excellent",
        "Good",
        "Poor"
    ]

    label_order = label_encoder.transform(class_order)

    print("=" * 70)
    print(f"{network_name} OOF Classification Report")
    print("=" * 70)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=label_order,
            target_names=class_order,
            digits=4,
            zero_division=0
        )
    )


display_classification_report(
    y_true=y_lte,
    y_pred=lte_oof_pred,
    label_encoder=le_lte,
    network_name="LTE"
)

print()

display_classification_report(
    y_true=y_nr,
    y_pred=nr_oof_pred,
    label_encoder=le_nr,
    network_name="5G NR"
)

In [ ]:
# ============================================================
# Cell 10. Poor-Class OOF Performance
# ============================================================

def get_poor_metrics(
    y_true,
    y_pred,
    label_encoder,
    network_name
):
    poor_label = label_encoder.transform(
        ["Poor"]
    )[0]

    poor_is_true = (
        y_true == poor_label
    )

    return {
        "Network": network_name,

        "Poor Precision": precision_score(
            y_true,
            y_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Recall": recall_score(
            y_true,
            y_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor F1": f1_score(
            y_true,
            y_pred,
            labels=[poor_label],
            average=None,
            zero_division=0
        )[0],

        "Poor Support": int(
            poor_is_true.sum()
        )
    }


poor_summary = pd.DataFrame([
    get_poor_metrics(
        y_true=y_lte,
        y_pred=lte_oof_pred,
        label_encoder=le_lte,
        network_name="LTE"
    ),

    get_poor_metrics(
        y_true=y_nr,
        y_pred=nr_oof_pred,
        label_encoder=le_nr,
        network_name="5G NR"
    )
])

print("Poor-Class OOF Performance")

display(
    poor_summary.round(4)
)

## 6. Out-of-Fold Confusion Matrices

Confusion matrices are generated separately for LTE and 5G NR using the pooled out-of-fold predictions.

The matrices show how frequently the Excellent, Good, and Poor classes are correctly classified or confused with one another.

### ※ Related dissertation sections

- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Figure 4.7 (LTE and 5G Pooled Out-of-Fold Confusion Matrices)
- Section 5.1 (Overall Classification Performance)

In [ ]:
# ============================================================
# Cell 11. LTE OOF Confusion Matrix
# ============================================================

class_order = [
    "Excellent",
    "Good",
    "Poor"
]

lte_label_order = le_lte.transform(
    class_order
)

lte_cm = confusion_matrix(
    y_lte,
    lte_oof_pred,
    labels=lte_label_order
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=lte_cm,
    display_labels=class_order
)

disp.plot(
    cmap="Blues",
    values_format="d",
    ax=ax,
    colorbar=True
)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

ax.set_xlabel(
    "Predicted class",
    fontsize=14
)

ax.set_ylabel(
    "True class",
    fontsize=14
)

ax.tick_params(
    axis="both",
    labelsize=13
)

ax.set_title(
    "LTE Three-Class OOF Confusion Matrix",
    fontsize=16,
    pad=12
)

plt.tight_layout()
plt.show()

print("LTE confusion matrix:")
display(
    pd.DataFrame(
        lte_cm,
        index=[
            f"True {label}"
            for label in class_order
        ],
        columns=[
            f"Predicted {label}"
            for label in class_order
        ]
    )
)

In [ ]:
lte_importance["Importance (%)"] = (
    lte_importance["Importance"]
    / lte_importance["Importance"].sum()
    * 100
)

## 7. LightGBM Feature Importance

Gain-based feature importance is averaged across the five cross-validation models.

The ten most important predictors are identified separately for LTE and 5G NR. Individual plots are produced first, followed by a combined high-resolution figure for the dissertation.

Feature importance represents contribution within the fitted models and should not be interpreted as evidence of a causal relationship.

### ※ Related dissertation sections

- Section 4.3 (Detailed Analysis of the Final Feature Set)
- Figure 4.8 (Top-10 LightGBM Feature Importances for LTE and 5G)
- Section 5.1 (Overall Classification Performance)

In [ ]:
# ============================================================
# Cell 12. LTE Top-10 Mean Feature Importance across 5 Folds
# ============================================================

lte_top10 = (
    lte_importance
    .head(10)
    .sort_values(
        "Importance (%)",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

bars = ax.barh(
    lte_top10["Feature"],
    lte_top10["Importance (%)"],
    color="steelblue"
)

# Add percentage labels
ax.bar_label(
    bars,
    labels=[
        f"{value:.1f}%"
        for value in lte_top10["Importance (%)"]
    ],
    padding=4,
    fontsize=10
)

ax.set_xlabel(
    "Mean LightGBM Gain Importance (%)",
    fontsize=12
)

ax.set_ylabel(
    "Feature",
    fontsize=12
)

ax.set_title(
    "LTE Top-10 Feature Importance across 5-Fold Cross-Validation",
    fontsize=14,
    pad=12
)

ax.tick_params(
    axis="both",
    labelsize=11
)

ax.set_xlim(
    0,
    lte_top10["Importance (%)"].max() * 1.18
)

plt.tight_layout()
plt.show()

display(
    lte_top10[
        ["Feature", "Importance", "Importance (%)"]
    ]
    .sort_values(
        "Importance (%)",
        ascending=False
    )
    .reset_index(drop=True)
    .round(3)
)

In [ ]:
# ============================================================
# Cell 13. 5G NR OOF Confusion Matrix
# ============================================================

class_order = [
    "Excellent",
    "Good",
    "Poor"
]

nr_label_order = le_nr.transform(
    class_order
)

nr_cm = confusion_matrix(
    y_nr,
    nr_oof_pred,
    labels=nr_label_order
)

fig, ax = plt.subplots(
    figsize=(7, 6)
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=nr_cm,
    display_labels=class_order
)

disp.plot(
    cmap="Greens",
    values_format="d",
    ax=ax,
    colorbar=True
)

# Increase numbers inside the matrix
for text in disp.text_.ravel():
    text.set_fontsize(16)

ax.set_xlabel(
    "Predicted class",
    fontsize=14
)

ax.set_ylabel(
    "True class",
    fontsize=14
)

ax.tick_params(
    axis="both",
    labelsize=13
)

ax.set_title(
    "5G NR Three-Class OOF Confusion Matrix",
    fontsize=16,
    pad=12
)

plt.tight_layout()
plt.show()

# Numerical table
nr_cm_table = pd.DataFrame(
    nr_cm,
    index=[
        f"True {label}"
        for label in class_order
    ],
    columns=[
        f"Predicted {label}"
        for label in class_order
    ]
)

print("5G NR confusion matrix:")
display(nr_cm_table)

In [ ]:
# ============================================================
# Cell 14. 5G NR Top-10 Mean Feature Importance across 5 Folds
# ============================================================

# Create percentage importance if it does not already exist
if "Importance (%)" not in nr_importance.columns:

    total_importance = nr_importance["Importance"].sum()

    nr_importance["Importance (%)"] = np.where(
        total_importance > 0,
        nr_importance["Importance"] / total_importance * 100,
        0
    )

# Select top 10 features
nr_top10 = (
    nr_importance
    .head(10)
    .sort_values(
        "Importance (%)",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

bars = ax.barh(
    nr_top10["Feature"],
    nr_top10["Importance (%)"],
    color="seagreen"
)

# Add percentage labels
ax.bar_label(
    bars,
    labels=[
        f"{value:.1f}%"
        for value in nr_top10["Importance (%)"]
    ],
    padding=4,
    fontsize=10
)

ax.set_xlabel(
    "Mean LightGBM Gain Importance (%)",
    fontsize=12
)

ax.set_ylabel(
    "Feature",
    fontsize=12
)

ax.set_title(
    "5G NR Top-10 Feature Importance across 5-Fold Cross-Validation",
    fontsize=14,
    pad=12
)

ax.tick_params(
    axis="both",
    labelsize=11
)

ax.set_xlim(
    0,
    nr_top10["Importance (%)"].max() * 1.18
)

plt.tight_layout()
plt.show()

# Numerical results
display(
    nr_top10[
        ["Feature", "Importance", "Importance (%)"]
    ]
    .sort_values(
        "Importance (%)",
        ascending=False
    )
    .reset_index(drop=True)
    .round(3)
)

In [ ]:
# ============================================================
# Figure 4.8. Top-10 LightGBM Gain Importances
# LTE and 5G NR
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Convert gain importance to percentage
# ------------------------------------------------------------

def prepare_top10_importance(importance_df):

    result = importance_df[
        ["Feature", "Importance"]
    ].copy()

    total_gain = result["Importance"].sum()

    if total_gain <= 0:
        raise ValueError(
            "Total feature importance must be greater than zero."
        )

    result["Importance (%)"] = (
        result["Importance"]
        / total_gain
        * 100
    )

    return (
        result
        .nlargest(
            10,
            "Importance (%)"
        )
        .sort_values(
            "Importance (%)",
            ascending=True
        )
    )


lte_top10 = prepare_top10_importance(
    lte_importance
)

nr_top10 = prepare_top10_importance(
    nr_importance
)


# ------------------------------------------------------------
# 2. Create combined figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6.5)
)

plot_settings = [
    (
        axes[0],
        lte_top10,
        "(a) LTE",
        "steelblue"
    ),
    (
        axes[1],
        nr_top10,
        "(b) 5G NR",
        "seagreen"
    )
]

for ax, data, title, colour in plot_settings:

    bars = ax.barh(
        data["Feature"],
        data["Importance (%)"],
        color=colour,
        alpha=0.85
    )

    ax.bar_label(
        bars,
        labels=[
            f"{value:.1f}%"
            for value in data["Importance (%)"]
        ],
        padding=4,
        fontsize=10
    )

    ax.set_title(
        title,
        fontsize=16,
        fontweight="bold",
        pad=10
    )

    ax.set_xlabel(
        "Mean Gain Importance (%)",
        fontsize=13
    )

    ax.tick_params(
        axis="x",
        labelsize=11
    )

    ax.tick_params(
        axis="y",
        labelsize=12
    )

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.25
    )

    ax.set_xlim(
        0,
        data["Importance (%)"].max() * 1.18
    )


# ------------------------------------------------------------
# 3. Main title and layout
# ------------------------------------------------------------

fig.suptitle(
    "Top-10 LightGBM Feature Importances",
    fontsize=17,
    fontweight="bold",
    y=0.99
)

plt.tight_layout(
    rect=[0, 0, 1, 0.94]
)


# ------------------------------------------------------------
# 4. High-resolution export
# ------------------------------------------------------------

figure_output_dir = str(FIGURES_DIR)

os.makedirs(
    figure_output_dir,
    exist_ok=True
)

figure_path = os.path.join(
    figure_output_dir,
    "figure_4_8_feature_importance.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Figure saved:")
print(figure_path)